In [9]:
import pandas as pd
import duckdb
from prefect import flow, task

In [10]:
datos = pd.read_excel(r"/workspaces/Proyecto_Aula_2/DATOS PROYECTO (1).xlsx")
datos = datos
datos = datos.astype(str)
datos.columns = [i.replace("/", "").replace("Ó", "O").replace("É", "E").replace(" ","_") for i in datos.columns]

In [11]:
#datos = datos

In [12]:
#datos = datos.astype(str)

In [13]:
datos.info()
datos.describe()

<class 'pandas.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   FECHA          251 non-null    str  
 1   EURUSD         251 non-null    str  
 2   INTERES_USA    251 non-null    str  
 3   INTERES_EUR    251 non-null    str  
 4   INFLACION_USA  251 non-null    str  
 5   INFLACION_EUR  251 non-null    str  
 6   PIB_USA        251 non-null    str  
 7   PIB_EUR        251 non-null    str  
dtypes: str(8)
memory usage: 15.8 KB


,FECHA,EURUSD,INTERES_USA,INTERES_EUR,INFLACION_USA,INFLACION_EUR,PIB_USA,PIB_EUR
count,251,251,251,251,251,251,251,251
unique,251,244,112,64,249,250,84,84
top,2005-02-01,1.3646,0.09,-0.4,0.0,0.0,12922.656,2293074.8
freq,1,2,20,41,3,2,3,3


In [14]:
#[i.replace("/", "").replace("Ó", "O").replace("É", "E").replace(" ","_") for i in datos.columns]

In [15]:
#datos.info()

In [16]:
@task
def generar_con():
    con = duckdb.connect("proyectolassupernenas.duckdb")
    return con


def crear_esquemas(con):
    con.execute("CREATE SCHEMA IF NOT EXISTS proyectolassupernenas.bronze")
    con.execute("CREATE SCHEMA IF NOT EXISTS proyectolassupernenas.silver")
    con.execute("CREATE SCHEMA IF NOT EXISTS proyectolassupernenas.gold")

def crear_tablas_precios_bronze(con):
    con.execute("""
CREATE OR REPLACE TABLE proyectolassupernenas.bronze.precios AS (
    SELECT * FROM datos
)
""")
    
def crear_tablas_precios_silver(con):
    con.execute("""
CREATE OR REPLACE TABLE proyectolassupernenas.silver.precios AS (
    SELECT 
        *,
        cast(INTERES_USA AS float) AS INTERES_USA_F,
    FROM proyectolassupernenas.bronze.precios
)
""")


In [24]:
@flow
def flujo_de_tareas():
    con = generar_con()
    tablas = crear_esquemas(con)

In [18]:
flujo_de_tareas()

22:43:50.281 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8370
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

22:43:56.799 | INFO    | Flow run 'outstanding-tuatara' - Beginning flow run 'outstanding-tuatara' for flow 'flujo-de-tareas'

22:43:57.131 | INFO    | Task run 'generar_con-7e0' - Finished in state Completed()

22:43:57.832 | INFO    | Flow run 'outstanding-tuatara' - Finished in state Completed()

In [28]:
import duckdb

con = duckdb.connect("proyectolassupernenas.duckdb")

In [29]:
con.execute("CREATE SCHEMA IF NOT EXISTS proyectolassupernenas.bronze")
con.execute("CREATE SCHEMA IF NOT EXISTS proyectolassupernenas.silver")
con.execute("CREATE SCHEMA IF NOT EXISTS proyectolassupernenas.gold")

In [30]:
con.execute("""
CREATE OR REPLACE TABLE proyectolassupernenas.bronze.precios AS (
    SELECT * FROM datos
)
""")

In [31]:
con.execute("""
CREATE OR REPLACE TABLE proyectolassupernenas.silver.precios AS (
    SELECT 
        *,
        cast("INTERÉS_USA" AS float) AS INTERES_USA_F
    FROM proyectolassupernenas.bronze.precios
)
""")

In [32]:
con.execute("""
CREATE OR REPLACE TABLE proyectolassupernenas.silver.precios AS (
    SELECT 
        *,

        cast("INTERÉS_USA" AS float) AS INTERES_USA_F,
        cast("INTERÉS_EUR" AS float) AS INTERES_EUR_F,   

        cast(INFLACION_USA AS float) AS INFLACION_USA_F,
        cast(INFLACION_EUR AS float) AS INFLACION_EUR_F,

        cast(PIB_USA AS float) AS PIB_USA_F,
        cast(PIB_EUR AS float) AS PIB_EUR_F,

        cast(FECHA AS date) AS FECHA_F,
        cast(EURUSD AS float) AS EURUSD_F

    FROM proyectolassupernenas.bronze.precios
)
""")

In [33]:
con.execute("""
CREATE OR REPLACE TABLE proyectolassupernenas.gold.modelo AS (
    SELECT
        FECHA_F,
        EURUSD_F,

        INTERES_USA_F - INTERES_EUR_F AS diff_rate,
        INFLACION_USA_F - INFLACION_EUR_F AS diff_inflation,
        PIB_USA_F - PIB_EUR_F AS diff_gdp

    FROM proyectolassupernenas.silver.precios
)
""")

In [34]:
@flow
def flujo_de_tareas():
    con = generar_con()
    crear_esquemas(con)
    crear_tablas_precios_bronze(con)
    crear_tablas_precios_silver(con)
    crear_tabla_gold(con)

In [35]:
import duckdb
con = duckdb.connect("proyectolassupernenas.duckdb")

# ¿existe gold?
con.execute("SHOW TABLES").df()

# ¿se ven bien los datos?
con.execute("""
SELECT * 
FROM proyectolassupernenas.gold.modelo
LIMIT 5
""").df()

# ¿hay nulos?
con.execute("""
SELECT 
    SUM(CASE WHEN diff_rate IS NULL THEN 1 ELSE 0 END) AS null_rate,
    SUM(CASE WHEN diff_inflation IS NULL THEN 1 ELSE 0 END) AS null_inf,
    SUM(CASE WHEN diff_gdp IS NULL THEN 1 ELSE 0 END) AS null_gdp
FROM proyectolassupernenas.gold.modelo
""").df()

,null_rate,null_inf,null_gdp
0,0.0,0.0,0.0


In [36]:
import duckdb

con = duckdb.connect("proyectolassupernenas.duckdb")

df = con.execute("""
SELECT * 
FROM proyectolassupernenas.gold.modelo
""").df()

df.head()

,FECHA_F,EURUSD_F,diff_rate,diff_inflation,diff_gdp
0,2005-02-01,1.3013,1.50,0.000570,-2260518.0
1,2005-03-01,1.3185,1.63,-0.003787,-2260518.0
2,2005-04-01,1.2943,1.79,-0.001054,-2280152.0
3,2005-05-01,1.2697,2.00,-0.002766,-2280152.0
4,2005-06-01,1.2155,2.04,-0.000547,-2280152.0


In [37]:
df[['EURUSD_F','diff_rate','diff_inflation','diff_gdp']].corr()

,EURUSD_F,diff_rate,diff_inflation,diff_gdp
EURUSD_F,1.000000,-0.482197,0.043930,0.673417
diff_rate,-0.482197,1.000000,0.057003,-0.360734
diff_inflation,0.043930,0.057003,1.000000,-0.015388
diff_gdp,0.673417,-0.360734,-0.015388,1.000000
